# Esperimenti di Prova - A2C CartPole e LunarLander

Questo notebook conserva il percorso sperimentale del vecchio `Lab3_A2C.ipynb`. Non è il notebook finale di consegna: serve a documentare quali tentativi sono stati fatti, quali settaggi hanno funzionato meglio e perché nel notebook finale si usa il checkpoint selezionato.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import torch
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from dla_lab3 import (
    a2c_from_env,
    evaluate_a2c_policy,
    load_a2c_checkpoint,
    lunar_a2c_from_env,
    make_env,
    observation_scale,
    set_seed,
    train_a2c_single_env,
    train_a2c_vectorized,
    A2CConfig,
)
from dla_lab3.paths import checkpoint_dir

SEED = 2112
set_seed(SEED)

## Esperimento 1 - Validazione A2C su CartPole

Vecchio risultato utile:

- checkpoint: `best_a2c_cartpole_gae_gamma098.pt`
- algoritmo: A2C con GAE
- `gamma=0.98`, `lr=7e-4`, `lr_critic=1e-3` nel vecchio codice
- risolto intorno all'episodio `425`
- valutazione greedy finale: media `500.0`, deviazione standard `0.0`, lunghezza media `500.0`

Questo esperimento valida che l'implementazione A2C sia corretta prima di passare a LunarLander.

In [ ]:
RUN_CARTPOLE_EXPERIMENT = False

if RUN_CARTPOLE_EXPERIMENT:
    train_env = make_env("CartPole-v1", seed=SEED)
    eval_env = make_env("CartPole-v1", seed=SEED + 1000)
    obs_scale = observation_scale("CartPole-v1")
    net = a2c_from_env(train_env, hidden_size=128)

    config = A2CConfig(
        gamma=0.98,
        lr=7e-4,
        value_coef=0.5,
        entropy_coef=0.001,
        entropy_coef_min=0.0,
        num_episodes=800,
        max_episode_steps=500,
        eval_every=25,
        eval_episodes=20,
        gae_lambda=0.95,
        normalize_advantage=True,
        grad_clip=0.5,
        solved_threshold=475.0,
        checkpoint_path=str(checkpoint_dir("best_a2c_cartpole_gae_gamma098.pt")),
        stop_when_solved=True,
    )

    history_cp = train_a2c_single_env(net, train_env, eval_env, obs_scale, config)
    train_env.close()
    eval_env.close()

    print("Solved:", history_cp["solved"])
    print("Best eval return:", history_cp["best_eval_return"])
else:
    print("CartPole experiment skipped. Set RUN_CARTPOLE_EXPERIMENT = True to rerun it.")

## Esperimenti LunarLander Scartati

Nel vecchio notebook sono stati provati diversi passaggi prima della soluzione finale:

| Tentativo | Settaggio | Risultato osservato |
|---|---|---|
| A2C vettoriale iniziale | GAE, reward scaling, rete stabile condivisa | media greedy circa `-140.48`, successo `0.0%` |
| A2C vettoriale stabilizzato | più stabilizzazioni ma architettura non ottimale | una valutazione è rimasta circa `-575.36` |
| A2C con actor/critic separati | `n_envs=16`, `n_steps=5`, `RMSprop`, `reward_scale=1.0`, `normalize_advantage=False` | netto miglioramento, ritorni positivi |
| Landing refinement | ripartenza dal checkpoint separato con learning rate più basso | migliore compromesso finale |

I primi tentativi non sono stati eliminati concettualmente: sono riportati qui come evidenza del percorso.

## Esperimento 2 - A2C LunarLander con Actor/Critic Separati

Settaggi principali del vecchio run:

- `n_envs=16`
- `gamma=0.995`
- `lr=8.3e-4`
- `entropy_coef=0.0005`, `entropy_coef_min=0.00001`
- `total_timesteps=1_000_000`
- `n_steps=5`
- `reward_scale=1.0`
- `gae_lambda=1.0`
- `optimizer=RMSprop`
- `normalize_advantage=False`
- checkpoint: `best_a2c_lunarlander_separate_seed2112.pt`

In [ ]:
RUN_LUNAR_SEPARATE = False

if RUN_LUNAR_SEPARATE:
    import gymnasium as gym

    def make_lunar_env(seed_value):
        def _factory():
            return make_env("LunarLander-v3", seed=seed_value, continuous=False, enable_wind=False)
        return _factory

    n_envs = 16
    vec_env = gym.vector.SyncVectorEnv([make_lunar_env(SEED + 5000 + i) for i in range(n_envs)])
    eval_env = make_env("LunarLander-v3", seed=SEED + 6000, continuous=False, enable_wind=False)
    obs_scale = torch.ones(8, dtype=torch.float32)
    net = lunar_a2c_from_env(eval_env, hidden_size=64)

    history_sep = train_a2c_vectorized(
        net,
        vec_env,
        eval_env,
        obs_scale,
        gamma=0.995,
        lr=8.3e-4,
        value_coef=0.5,
        entropy_coef=0.0005,
        entropy_coef_min=0.00001,
        total_timesteps=1_000_000,
        n_steps=5,
        eval_every=250,
        eval_episodes=50,
        reward_scale=1.0,
        gae_lambda=1.0,
        optimizer_name="rmsprop",
        checkpoint_path=str(checkpoint_dir("best_a2c_lunarlander_separate_seed2112.pt")),
        normalize_advantage=False,
        grad_clip=0.5,
        stop_when_solved=False,
    )

    vec_env.close()
    eval_env.close()
    print("Best eval return:", history_sep["best_eval_return"])
else:
    print("Separate LunarLander experiment skipped.")

## Esperimento 3 - Landing Refinement

Questo run riparte dal checkpoint separato e abbassa il learning rate per migliorare il comportamento di atterraggio senza distruggere la policy già appresa.

Settaggi principali:

- checkpoint iniziale: `best_a2c_lunarlander_separate_seed2112.pt`
- `lr=1.0e-4`
- `entropy_coef=0.00008`
- `total_timesteps=300_000`
- `eval_episodes=100`
- checkpoint finale: `best_a2c_lunarlander_landing_refine_seed2112.pt`

In [ ]:
RUN_LANDING_REFINE = False

if RUN_LANDING_REFINE:
    import gymnasium as gym

    def make_lunar_env(seed_value):
        def _factory():
            return make_env("LunarLander-v3", seed=seed_value, continuous=False, enable_wind=False)
        return _factory

    n_envs = 16
    vec_env = gym.vector.SyncVectorEnv([make_lunar_env(SEED + 7000 + i) for i in range(n_envs)])
    eval_env = make_env("LunarLander-v3", seed=SEED + 8000, continuous=False, enable_wind=False)
    obs_scale = torch.ones(8, dtype=torch.float32)
    net = lunar_a2c_from_env(eval_env, hidden_size=64)

    initial_checkpoint = checkpoint_dir("old", "best_a2c_lunarlander_separate_seed2112.pt", create=False)
    if initial_checkpoint.exists():
        load_a2c_checkpoint(net, initial_checkpoint)
        print("Loaded:", initial_checkpoint)
    else:
        print("Initial checkpoint not found:", initial_checkpoint)

    history_refine = train_a2c_vectorized(
        net,
        vec_env,
        eval_env,
        obs_scale,
        gamma=0.995,
        lr=1.0e-4,
        value_coef=0.5,
        entropy_coef=0.00008,
        entropy_coef_min=0.00001,
        total_timesteps=300_000,
        n_steps=5,
        eval_every=250,
        eval_episodes=100,
        reward_scale=1.0,
        gae_lambda=1.0,
        optimizer_name="rmsprop",
        checkpoint_path=str(checkpoint_dir("best_a2c_lunarlander_landing_refine_seed2112.pt")),
        normalize_advantage=False,
        grad_clip=0.5,
        stop_when_solved=False,
    )

    vec_env.close()
    eval_env.close()
    print("Best eval return:", history_refine["best_eval_return"])
else:
    print("Landing refinement skipped.")

## Confronto Finale Checkpoint

Risultati conservati dal vecchio notebook, valutati su 200 episodi:

| Checkpoint | Greedy avg | Greedy success | Stochastic avg | Stochastic success |
|---|---:|---:|---:|---:|
| `best_a2c_lunarlander_separate_seed2112.pt` | `99.97` | `13.5%` | `100.57` | `24.0%` |
| `best_a2c_lunarlander_landing_refine_seed2112.pt` | `125.84` | `12.0%` | `127.61` | `27.5%` |
| `best_a2c_lunarlander_landing_refine_seed2112_best_train.pt` | `126.88` | `13.0%` | `132.43` | `31.0%` |
| `best_a2c_lunarlander_landing_refine_seed2112_last.pt` | `121.46` | `10.5%` | `140.16` | `29.5%` |

## Temperature Sweep

Risultati conservati dal vecchio notebook con checkpoint `best_a2c_lunarlander_landing_refine_seed2112_best_train.pt`:

| Temperature | Avg return | Std return | Success >= 200 |
|---:|---:|---:|---:|
| `0.25` | `111.45` | `106.66` | `15.5%` |
| `0.50` | `124.24` | `99.97` | `21.5%` |
| `0.75` | `129.27` | `102.50` | `27.5%` |
| `1.00` | `141.03` | `101.20` | `34.5%` |
| `1.25` | `139.99` | `102.77` | `39.5%` |

Scelta finale: temperatura `1.0` se si privilegia il ritorno medio; temperatura `1.25` se si privilegia solo la percentuale di episodi sopra `200`.

## Conclusione degli Esperimenti

Il percorso sperimentale mostra che:

- CartPole viene risolto e quindi A2C è implementato correttamente;
- LunarLander non viene risolto stabilmente secondo la soglia media `>= 200`;
- la policy finale è comunque molto migliore dei tentativi iniziali;
- la soluzione migliore osservata richiede actor/critic separati, rollout vettoriali, `RMSprop`, GAE con `lambda=1.0`, learning rate più basso nel refinement e valutazione stocastica controllata dalla temperatura.

## Referenced functions and source files

| Function/class | Defined in | Purpose |
| --- | --- | --- |
| `reinforce` / `reinforce_with_value_baseline` | `src/dla_lab3/policy_gradient.py` | Training policy-gradient su CartPole. |
| `A2CConfig` / `train_a2c_vectorized` | `src/dla_lab3/a2c.py` | Configurazione e training A2C. |
| `evaluate_lunar_candidates` | `src/dla_lab3/experiments.py` | Valutazione checkpoint LunarLander. |
| `run_lunar_visual_episodes` | `src/dla_lab3/visualization.py` | Rollout visuali finali. |
